# 13 · Distance Metrics and Bootstrap Uncertainty

This notebook quantifies the geometric similarity story from Notebooks 10–12.

We compare zeta, Poisson, and GUE local-window descriptor clouds using:

1. centroid distances
2. simple 2D density-grid distances
3. 1D Wasserstein-style distances on PCA coordinates
4. a symmetrized density divergence on 2D grids
5. bootstrap uncertainty intervals for the main distance quantities

## Goal

The key question is:

> Is zeta closer to GUE than to Poisson, and how stable is that conclusion under resampling?

This notebook keeps everything lightweight and reproducible:
- no heavy external ML libraries
- bootstrap done directly with NumPy
- distances computed from the same descriptor pipeline used earlier

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources and descriptor pipeline

In [ ]:
# Zeta gaps
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r)), float(np.std(r))

def descriptor_vector(window):
    ratio_mean, ratio_std = ratio_features(window)
    return np.array([
        window[0], window[1], window[2], window[3], window[4],
        unevenness(window),
        reversal_symmetry_score(window),
        window_entropy(window),
        ratio_mean,
        ratio_std,
    ], dtype=float)

def build_descriptors(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    desc = np.array([descriptor_vector(w) for w in windows_n])
    return windows_n, desc

zeta_w, zeta_desc = build_descriptors(zeta_gaps, k=5)
poisson_w, poisson_desc = build_descriptors(poisson_gaps, k=5)
gue_w, gue_desc = build_descriptors(gue_gaps, k=5)

## Standardize and embed with PCA

In [ ]:
all_desc = np.vstack([zeta_desc, poisson_desc, gue_desc])

mean = all_desc.mean(axis=0)
std = all_desc.std(axis=0)
std[std == 0] = 1.0

zeta_std = (zeta_desc - mean) / std
poisson_std = (poisson_desc - mean) / std
gue_std = (gue_desc - mean) / std

X = np.vstack([zeta_std, poisson_std, gue_std])
Xc = X - X.mean(axis=0)

cov = np.cov(Xc, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

components = eigvecs[:, :2]
embedding = Xc @ components

n_zeta = len(zeta_std)
n_poisson = len(poisson_std)
n_gue = len(gue_std)

zeta_emb = embedding[:n_zeta]
poisson_emb = embedding[n_zeta:n_zeta + n_poisson]
gue_emb = embedding[n_zeta + n_poisson:]

## Quick PCA overview

In [ ]:
plt.figure(figsize=(8.8, 6.2))
plt.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=10, alpha=0.25, label="zeta")
plt.scatter(poisson_emb[:, 0], poisson_emb[:, 1], s=10, alpha=0.25, label="Poisson")
plt.scatter(gue_emb[:, 0], gue_emb[:, 1], s=10, alpha=0.25, label="GUE")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Embedding overview for distance analysis")
plt.legend()
plt.show()

## Distance helpers

In [ ]:
def centroid_distance(A, B):
    return float(np.linalg.norm(A.mean(axis=0) - B.mean(axis=0)))

def density_grid(X, bins_x, bins_y):
    H, _, _ = np.histogram2d(X[:, 0], X[:, 1], bins=[bins_x, bins_y], density=True)
    return H

def grid_l2_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

def sym_kl_divergence(P, Q, eps=1e-12):
    P = P.ravel() + eps
    Q = Q.ravel() + eps
    P = P / P.sum()
    Q = Q / Q.sum()
    return float(np.sum(P * np.log(P / Q)) + np.sum(Q * np.log(Q / P)))

def wasserstein_1d(a, b):
    a = np.sort(np.asarray(a))
    b = np.sort(np.asarray(b))
    n = min(len(a), len(b))
    if len(a) != n:
        idx = np.linspace(0, len(a) - 1, n).astype(int)
        a = a[idx]
    if len(b) != n:
        idx = np.linspace(0, len(b) - 1, n).astype(int)
        b = b[idx]
    return float(np.mean(np.abs(a - b)))

## Base distance metrics

In [ ]:
xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

bins_x = np.linspace(xmin, xmax, 36)
bins_y = np.linspace(ymin, ymax, 36)

zeta_H = density_grid(zeta_emb, bins_x, bins_y)
poisson_H = density_grid(poisson_emb, bins_x, bins_y)
gue_H = density_grid(gue_emb, bins_x, bins_y)

base_metrics = {
    "centroid_distance": {
        "zeta_GUE": centroid_distance(zeta_emb, gue_emb),
        "zeta_Poisson": centroid_distance(zeta_emb, poisson_emb),
        "GUE_Poisson": centroid_distance(gue_emb, poisson_emb),
    },
    "grid_l2_distance": {
        "zeta_GUE": grid_l2_distance(zeta_H, gue_H),
        "zeta_Poisson": grid_l2_distance(zeta_H, poisson_H),
        "GUE_Poisson": grid_l2_distance(gue_H, poisson_H),
    },
    "sym_kl_divergence": {
        "zeta_GUE": sym_kl_divergence(zeta_H, gue_H),
        "zeta_Poisson": sym_kl_divergence(zeta_H, poisson_H),
        "GUE_Poisson": sym_kl_divergence(gue_H, poisson_H),
    },
    "wasserstein_pc1": {
        "zeta_GUE": wasserstein_1d(zeta_emb[:, 0], gue_emb[:, 0]),
        "zeta_Poisson": wasserstein_1d(zeta_emb[:, 0], poisson_emb[:, 0]),
        "GUE_Poisson": wasserstein_1d(gue_emb[:, 0], poisson_emb[:, 0]),
    },
    "wasserstein_pc2": {
        "zeta_GUE": wasserstein_1d(zeta_emb[:, 1], gue_emb[:, 1]),
        "zeta_Poisson": wasserstein_1d(zeta_emb[:, 1], poisson_emb[:, 1]),
        "GUE_Poisson": wasserstein_1d(gue_emb[:, 1], poisson_emb[:, 1]),
    },
}
base_metrics

## Visualize base distances

In [ ]:
for metric_name, metric_vals in base_metrics.items():
    labels = list(metric_vals.keys())
    vals = [metric_vals[k] for k in labels]

    plt.figure(figsize=(7.8, 4.6))
    plt.bar(labels, vals)
    plt.ylabel(metric_name)
    plt.title(f"{metric_name} comparison")
    plt.show()

## Bootstrap uncertainty

We resample windows within each ensemble and recompute the metrics.
This gives rough uncertainty intervals for the key comparisons.

In [ ]:
def bootstrap_sample_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_metrics(zeta_X, poisson_X, gue_X, bins_x, bins_y):
    zeta_H = density_grid(zeta_X, bins_x, bins_y)
    poisson_H = density_grid(poisson_X, bins_x, bins_y)
    gue_H = density_grid(gue_X, bins_x, bins_y)

    return {
        "centroid_distance_zeta_GUE": centroid_distance(zeta_X, gue_X),
        "centroid_distance_zeta_Poisson": centroid_distance(zeta_X, poisson_X),
        "grid_l2_zeta_GUE": grid_l2_distance(zeta_H, gue_H),
        "grid_l2_zeta_Poisson": grid_l2_distance(zeta_H, poisson_H),
        "sym_kl_zeta_GUE": sym_kl_divergence(zeta_H, gue_H),
        "sym_kl_zeta_Poisson": sym_kl_divergence(zeta_H, poisson_H),
        "wasserstein_pc1_zeta_GUE": wasserstein_1d(zeta_X[:, 0], gue_X[:, 0]),
        "wasserstein_pc1_zeta_Poisson": wasserstein_1d(zeta_X[:, 0], poisson_X[:, 0]),
        "wasserstein_pc2_zeta_GUE": wasserstein_1d(zeta_X[:, 1], gue_X[:, 1]),
        "wasserstein_pc2_zeta_Poisson": wasserstein_1d(zeta_X[:, 1], poisson_X[:, 1]),
    }

boot_rng = np.random.default_rng(9423)
n_boot = 80

boot_records = []
for _ in range(n_boot):
    zb = bootstrap_sample_rows(zeta_emb, boot_rng)
    pb = bootstrap_sample_rows(poisson_emb, boot_rng)
    gb = bootstrap_sample_rows(gue_emb, boot_rng)
    boot_records.append(bootstrap_metrics(zb, pb, gb, bins_x, bins_y))

In [ ]:
metric_names = list(boot_records[0].keys())

bootstrap_summary = {}
for name in metric_names:
    vals = np.array([r[name] for r in boot_records])
    bootstrap_summary[name] = {
        "mean": float(vals.mean()),
        "std": float(vals.std()),
        "q05": float(np.quantile(vals, 0.05)),
        "q95": float(np.quantile(vals, 0.95)),
    }

bootstrap_summary

## Bootstrap interval plots

We compare zeta–GUE and zeta–Poisson side by side for several metrics.

In [ ]:
metric_pairs = [
    ("centroid_distance_zeta_GUE", "centroid_distance_zeta_Poisson", "centroid distance"),
    ("grid_l2_zeta_GUE", "grid_l2_zeta_Poisson", "grid L2 distance"),
    ("sym_kl_zeta_GUE", "sym_kl_zeta_Poisson", "symmetrized KL"),
    ("wasserstein_pc1_zeta_GUE", "wasserstein_pc1_zeta_Poisson", "Wasserstein PC1"),
    ("wasserstein_pc2_zeta_GUE", "wasserstein_pc2_zeta_Poisson", "Wasserstein PC2"),
]

for m_gue, m_pois, title in metric_pairs:
    labels = ["zeta–GUE", "zeta–Poisson"]
    means = [bootstrap_summary[m_gue]["mean"], bootstrap_summary[m_pois]["mean"]]
    low = [bootstrap_summary[m_gue]["mean"] - bootstrap_summary[m_gue]["q05"],
           bootstrap_summary[m_pois]["mean"] - bootstrap_summary[m_pois]["q05"]]
    high = [bootstrap_summary[m_gue]["q95"] - bootstrap_summary[m_gue]["mean"],
            bootstrap_summary[m_pois]["q95"] - bootstrap_summary[m_pois]["mean"]]

    x = np.arange(2)
    plt.figure(figsize=(6.8, 4.4))
    plt.bar(x, means, yerr=np.array([low, high]), capsize=5)
    plt.xticks(x, labels)
    plt.ylabel(title)
    plt.title(f"Bootstrap comparison: {title}")
    plt.show()

## Difference-of-distance summaries

A very direct question is whether

\[
d(\text{zeta}, \text{Poisson}) - d(\text{zeta}, \text{GUE})
\]

is stably positive.

In [ ]:
difference_summary = {
    "centroid_gap": {
        "mean": bootstrap_summary["centroid_distance_zeta_Poisson"]["mean"] - bootstrap_summary["centroid_distance_zeta_GUE"]["mean"],
        "q05": bootstrap_summary["centroid_distance_zeta_Poisson"]["q05"] - bootstrap_summary["centroid_distance_zeta_GUE"]["q95"],
        "q95": bootstrap_summary["centroid_distance_zeta_Poisson"]["q95"] - bootstrap_summary["centroid_distance_zeta_GUE"]["q05"],
    },
    "grid_l2_gap": {
        "mean": bootstrap_summary["grid_l2_zeta_Poisson"]["mean"] - bootstrap_summary["grid_l2_zeta_GUE"]["mean"],
        "q05": bootstrap_summary["grid_l2_zeta_Poisson"]["q05"] - bootstrap_summary["grid_l2_zeta_GUE"]["q95"],
        "q95": bootstrap_summary["grid_l2_zeta_Poisson"]["q95"] - bootstrap_summary["grid_l2_zeta_GUE"]["q05"],
    },
    "sym_kl_gap": {
        "mean": bootstrap_summary["sym_kl_zeta_Poisson"]["mean"] - bootstrap_summary["sym_kl_zeta_GUE"]["mean"],
        "q05": bootstrap_summary["sym_kl_zeta_Poisson"]["q05"] - bootstrap_summary["sym_kl_zeta_GUE"]["q95"],
        "q95": bootstrap_summary["sym_kl_zeta_Poisson"]["q95"] - bootstrap_summary["sym_kl_zeta_GUE"]["q05"],
    },
}
difference_summary

## Final summary

In [ ]:
summary = {
    "base_metrics": base_metrics,
    "bootstrap_summary": bootstrap_summary,
    "difference_summary": difference_summary,
}
summary

## Notes

- Smaller zeta–GUE distances than zeta–Poisson distances across several metrics support the interpretation that zeta and GUE share a common local descriptor geometry.
- Bootstrap intervals help show whether the conclusion is stable under resampling of local windows.
- The distance measures here are intentionally lightweight; later notebooks could add more formal transport or kernel-based distances.

## Next directions

A natural next notebook could do one of these:

1. extend the bootstrap analysis to conditional windows (after small vs after large gaps)  
2. compare multiple window lengths (3-gap, 5-gap, 7-gap)  
3. add a classifier-confidence / calibration analysis  
4. test descriptor robustness under modified feature sets